In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [13]:
import pprint

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [14]:
# Q1. How many lesson pages
print(len(documents))

72


In [4]:
# Q2. Indexing and searching
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query=question,
    num_results=5,
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [9]:
# Q3. RAG
from rag_helper import RAGBase
from openai import OpenAI

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results,
        )
        
    
    def build_context(self, search_results):
        lines = []

        for file in search_results:
            lines.append("filename: " + file["filename"])
            lines.append("Content: " + file["content"])
            lines.append("")

        return "\n".join(lines).strip()
    
openai_client = OpenAI()

assistant = LessonRAG(
    index=index, llm_client=openai_client
)
    
question = "How does the agentic loop keep calling the model until it stops?"

answer, usage = assistant.rag(question)
usage

7136

In [15]:
# Q4. Chunking

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)


295

In [18]:
# Q5. RAG with chunking
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

assistant = LessonRAG(
    index=chunk_index, llm_client=openai_client
)
    
question = "How does the agentic loop keep calling the model until it stops?"

answer, usage = assistant.rag(question)
usage


2319

In [ ]:
# Q6. Turning it into an agent
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0
)

@tool
def search(query: str):
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
    )
system_prompt = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

agent = create_agent(
    model=llm,
    tools=[search],
    system_prompt=system_prompt
)

results = agent.invoke(
    {"messages": [
        {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
    ]}
)

tool_calls = 0 
for message in results["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        tool_calls += len(message.tool_calls)

tool_calls

4